Efficiency

Load BESStie

In [ ]:
# a lot of ignore warnings were creating noisy outputs, so filter these away.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Load BESSTIE dataset and view overall stats
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

print(dataset)

training_dataframe = dataset['train'].to_pandas()
test_dataframe = dataset['test'].to_pandas()

In [ ]:
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = model.to(device)
model.eval()

def inference_timings(model, tokenizer, texts, device, n_runs=50):
    times = []
    for _ in range(n_runs):
        start = time.time()
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elif device.type == "mps":
            torch.mps.synchronize()
        times.append((time.time() - start) * 1000)
    return np.mean(times), np.std(times)

test_texts = test_dataframe['text'].tolist()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
elif torch.backends.mps.is_available():
    print(f"GPU: Apple MPS")
else:
    print("GPU: None")

# Warmup
dummy = tokenizer(test_texts[:1], return_tensors="pt", padding=True, truncation=True, max_length=128)
dummy = {k: v.to(device) for k, v in dummy.items()}
with torch.no_grad():
    model(**dummy)
if device.type == "cuda":
    torch.cuda.synchronize()
elif device.type == "mps":
    torch.mps.synchronize()

# Real run
for label, texts in [
    ("Single", test_texts[:1]),
    ("Batch 16", test_texts[:16]),
    ("Batch 32", test_texts[:32]),
    ("Batch 64", test_texts[:64]),
    ("Batch 128", test_texts[:128]),
    ("Batch 256", test_texts[:256]),
    ("Batch 512", test_texts[:512]),
    ("Batch 1024", test_texts[:1024]),
]:
    mean_ms, std_ms = inference_timings(model, tokenizer, texts, device)
    print(f"{label:>10}: {mean_ms:.2f} ± {std_ms:.2f} ms")


Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])

lr_sentiment = LogisticRegression(max_iter=1000)
lr_sentiment.fit(X_train, training_dataframe['Sentiment'])

vectorizer.transform(test_texts[:1])
lr_sentiment.predict(vectorizer.transform(test_texts[:1]))

def lr_timings(model, vectorizer, texts, n_runs=50):
    times = []
    for _ in range(n_runs):
        start = time.time()
        X = vectorizer.transform(texts)
        model.predict(X)
        times.append((time.time() - start) * 1000)
    return np.mean(times), np.std(times)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
elif torch.backends.mps.is_available():
    print(f"GPU: Apple MPS")
else:
    print("GPU: None")

for label, texts in [
    ("Single", test_texts[:1]),
    ("Batch 16", test_texts[:16]),
    ("Batch 32", test_texts[:32]),
    ("Batch 64", test_texts[:64]),
    ("Batch 128", test_texts[:128]),
    ("Batch 256", test_texts[:256]),
    ("Batch 512", test_texts[:512]),
    ("Batch 1024", test_texts[:1024]),
]:
    lr_mean_ms, lr_std_ms = lr_timings(lr_sentiment, vectorizer, texts)
    print(f"{label:>10}: {lr_mean_ms:.2f} ± {lr_std_ms:.2f} ms")
    
ratio = mean_ms / lr_mean_ms    
print(f"RoBERTa to LR ratio for Batch 1024: {ratio:.2f}x")